# Module 04: Comparing Groups & ANOVA
*Statistics for Econometrics — An intermediate course for researchers*

This module covers methods for comparing means across groups: two-sample t-tests, paired t-tests, ANOVA, and post-hoc analysis. We use examples from regional economics and logistics infrastructure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# For post-hoc tests
try:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
except ImportError:
    !pip install statsmodels
    from statsmodels.stats.multicomp import pairwise_tukeyhsd

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 4.1 The Two-Sample Problem

Often in econometrics we wish to compare the mean outcome in two different groups. For example:
- Does proximity to an interporto (inland port) affect municipal property values?
- Do two regions have significantly different average household incomes?

### The t-test statistic

We have two independent random samples: $X_1, \ldots, X_{n_1}$ from group 1 and $Y_1, \ldots, Y_{n_2}$ from group 2. Under the null hypothesis $H_0: \mu_1 = \mu_2$, the test statistic is:

$$t = \frac{\bar{X} - \bar{Y}}{SE(\bar{X} - \bar{Y})}$$

The standard error of the difference in means is:

$$SE(\bar{X} - \bar{Y}) = \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}$$

where $s_1^2$ and $s_2^2$ are the sample variances. Under $H_0$, $t$ follows an approximate $t$-distribution with degrees of freedom given by the Welch–Satterthwaite equation (which does not assume equal variances). For equal variances, we can pool the estimates, but modern practice uses the Welch correction.

In [ ]:
# Two-sample t-test: Municipality income, near vs far from interporto
# (in thousands of euros per capita)

np.random.seed(42)

# Group A: Municipalities near interporto
n_A = 10
mean_A = 21.4  # £21,400
sd_A = 2.8
group_A = np.random.normal(mean_A, sd_A, n_A)

# Group B: Control municipalities (far from interporto)
n_B = 10
mean_B = 23.1  # £23,100
sd_B = 3.2
group_B = np.random.normal(mean_B, sd_B, n_B)

print("Group A (Near Interporto):")
print(f"  Data: {group_A.round(2)}")
print(f"  Sample mean: £{group_A.mean():.2f}k")
print(f"  Sample SD: £{group_A.std(ddof=1):.2f}k")
print(f"  n = {n_A}\n")

print("Group B (Control):")
print(f"  Data: {group_B.round(2)}")
print(f"  Sample mean: £{group_B.mean():.2f}k")
print(f"  Sample SD: £{group_B.std(ddof=1):.2f}k")
print(f"  n = {n_B}\n")

# Manual calculation
s_A = group_A.std(ddof=1)
s_B = group_B.std(ddof=1)
se_diff = np.sqrt((s_A**2 / n_A) + (s_B**2 / n_B))
t_stat = (group_A.mean() - group_B.mean()) / se_diff

print("Manual Calculation:")
print(f"  s_A² = {s_A**2:.4f}, s_B² = {s_B**2:.4f}")
print(f"  SE(diff) = √({s_A**2:.4f}/{n_A} + {s_B**2:.4f}/{n_B}) = {se_diff:.4f}")
print(f"  t = ({group_A.mean():.4f} - {group_B.mean():.4f}) / {se_diff:.4f} = {t_stat:.4f}\n")

# Using scipy (Welch's t-test, does not assume equal variances)
t_scipy, p_value = stats.ttest_ind(group_A, group_B, equal_var=False)
print("Scipy (Welch's t-test):")
print(f"  t = {t_scipy:.4f}")
print(f"  p-value = {p_value:.4f}")
print(f"  Conclusion: {'Reject H0 (p < 0.05)' if p_value < 0.05 else 'Fail to reject H0 (p >= 0.05)'}")

In [ ]:
# Visualisation: Box plots and violin plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Prepare data for plotting
data_for_plot = pd.DataFrame({
    'Income (£k)': np.concatenate([group_A, group_B]),
    'Group': ['Near Interporto']*n_A + ['Control']*n_B
})

# Box plot
sns.boxplot(data=data_for_plot, x='Group', y='Income (£k)', ax=axes[0], palette='Set2')
axes[0].set_title('Box Plot: Municipality Income by Group', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Income (£ thousands)')
axes[0].axhline(y=group_A.mean(), color='C0', linestyle='--', alpha=0.5, label='Mean (Near)')
axes[0].axhline(y=group_B.mean(), color='C1', linestyle='--', alpha=0.5, label='Mean (Control)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Violin plot
sns.violinplot(data=data_for_plot, x='Group', y='Income (£k)', ax=axes[1], palette='Set2')
axes[1].set_title('Violin Plot: Distribution by Group', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Income (£ thousands)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInterpretation: The group near the interporto has a mean income £{group_B.mean() - group_A.mean():.2f}k lower than control.")
print(f"This difference is {'statistically significant' if p_value < 0.05 else 'not statistically significant'} at the 5% level.")

## 4.2 Paired Samples

When the same units (municipalities, households, firms) are measured twice—before and after an intervention, or under two different conditions—we have **paired data**. The key insight is that we can eliminate between-unit variation by working with the differences.

### The paired t-test

Let $d_i = X_i - Y_i$ be the difference for unit $i$. Then:

$$t = \frac{\bar{d}}{SE(\bar{d})} = \frac{\bar{d}}{s_d / \sqrt{n}}$$

This is simply a one-sample t-test applied to the differences. The paired test is more powerful than the independent t-test because it accounts for the within-unit correlation, reducing the variance of the test statistic.

**Example:** Property values in 20 municipalities before and after a new logistics hub opens nearby.

In [ ]:
# Paired t-test: Property values before and after interporto development
# (in thousands of pounds)

np.random.seed(42)

n_pairs = 20

# Before: varied baseline values across municipalities
before = np.random.normal(180, 30, n_pairs)

# After: small decrease after hub opens (on average)
# Each municipality's change is correlated with its baseline
change = np.random.normal(-5, 8, n_pairs)
after = before + change

differences = after - before

print("Paired Data Summary:")
print(f"  Mean before: £{before.mean():.2f}k")
print(f"  Mean after:  £{after.mean():.2f}k")
print(f"  Mean difference: £{differences.mean():.2f}k")
print(f"  SD of differences: £{differences.std(ddof=1):.2f}k\n")

# Paired t-test
t_paired, p_paired = stats.ttest_rel(after, before)
print("Paired t-test:")
print(f"  t = {t_paired:.4f}")
print(f"  p-value = {p_paired:.4f}")
print(f"  df = {n_pairs - 1}")
print(f"  Conclusion: {'Reject H0' if p_paired < 0.05 else 'Fail to reject H0'} at 5% level\n")

# For comparison: incorrect independent t-test (ignoring pairing)
t_indep, p_indep = stats.ttest_ind(after, before, equal_var=False)
print("Independent t-test (INCORRECT—ignores pairing):")
print(f"  t = {t_indep:.4f}")
print(f"  p-value = {p_indep:.4f}")
print(f"\nNote: The paired test is more powerful because it exploits the correlation within pairs.")

In [ ]:
# Visualisation: Before-After with paired lines
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Scatter plot with connecting lines
ax = axes[0]
x_before = np.zeros(n_pairs)
x_after = np.ones(n_pairs)

for i in range(n_pairs):
    ax.plot([x_before[i], x_after[i]], [before[i], after[i]], 'C0-', alpha=0.3, linewidth=1)

ax.scatter(x_before, before, s=80, alpha=0.6, label='Before', color='C0', zorder=3)
ax.scatter(x_after, after, s=80, alpha=0.6, label='After', color='C1', zorder=3)

# Overlay means
ax.scatter([0], [before.mean()], s=200, marker='D', color='C0', edgecolor='black', 
          linewidth=2, zorder=4, label='Mean (Before)')
ax.scatter([1], [after.mean()], s=200, marker='D', color='C1', edgecolor='black', 
          linewidth=2, zorder=4, label='Mean (After)')

ax.set_xticks([0, 1])
ax.set_xticklabels(['Before', 'After'])
ax.set_ylabel('Property Value (£ thousands)')
ax.set_title('Paired Observations: Before-After Property Values', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Distribution of differences
ax = axes[1]
ax.hist(differences, bins=8, edgecolor='black', alpha=0.7, color='C2')
ax.axvline(differences.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = £{differences.mean():.2f}k')
ax.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5, label='Zero (H₀)')
ax.set_xlabel('Change in Property Value (£ thousands)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Differences (After - Before)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4.3 The Multiple Comparisons Problem

When we conduct multiple statistical tests, the **family-wise Type I error rate** (probability of at least one false positive) grows rapidly. If we run $m$ independent tests, each at level $\alpha$, then:

$$P(\text{at least one Type I error}) = 1 - (1 - \alpha)^m \approx m \cdot \alpha \quad \text{(for small } \alpha \text{)}$$

For example:
- 2 tests at $\alpha = 0.05$: error rate ≈ 0.098
- 3 tests at $\alpha = 0.05$: error rate ≈ 0.143
- 5 tests at $\alpha = 0.05$: error rate ≈ 0.226

This is why we use **corrections** (Bonferroni, Tukey HSD, etc.) or **omnibus tests** (ANOVA before pairwise comparisons).

In [ ]:
# Simulation: False positive rate under multiple comparisons
# H₀ is true: all groups come from the same distribution

np.random.seed(42)

n_sims = 10000
n_per_group = 20
alpha = 0.05

# Test different numbers of groups
n_groups_list = [2, 3, 5, 10]
observed_error_rates = []
expected_error_rates = []

for n_groups in n_groups_list:
    # Count simulations with at least one significant pairwise comparison
    n_false_positives = 0
    n_comparisons = n_groups * (n_groups - 1) // 2  # Number of pairwise tests
    
    for sim in range(n_sims):
        # Generate all groups from same distribution
        groups = [np.random.normal(100, 15, n_per_group) for _ in range(n_groups)]
        
        # Perform all pairwise t-tests
        found_sig = False
        for i in range(n_groups):
            for j in range(i + 1, n_groups):
                _, p = stats.ttest_ind(groups[i], groups[j], equal_var=False)
                if p < alpha:  # No correction applied
                    found_sig = True
                    break
            if found_sig:
                break
        
        if found_sig:
            n_false_positives += 1
    
    observed_rate = n_false_positives / n_sims
    expected_rate = 1 - (1 - alpha)**n_comparisons  # Theoretical rate
    
    observed_error_rates.append(observed_rate)
    expected_error_rates.append(expected_rate)
    
    print(f"\nWith {n_groups} groups ({n_comparisons} pairwise comparisons):")
    print(f"  Observed false positive rate: {observed_rate:.3f}")
    print(f"  Expected (1 - (1 - α)^m):    {expected_rate:.3f}")
    print(f"  Nominal α:                    {alpha:.3f}")

In [ ]:
# Visualise the inflation of Type I error
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(n_groups_list))
width = 0.35

bars1 = ax.bar(x - width/2, observed_error_rates, width, label='Observed (from simulation)', 
               color='C0', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, expected_error_rates, width, label='Expected (theoretical)', 
               color='C1', alpha=0.8, edgecolor='black')

ax.axhline(y=alpha, color='red', linestyle='--', linewidth=2, label=f'Nominal α = {alpha}')

ax.set_xlabel('Number of Groups', fontsize=11, fontweight='bold')
ax.set_ylabel('Family-Wise Type I Error Rate', fontsize=11, fontweight='bold')
ax.set_title('Multiple Comparisons Problem: Type I Error Inflation\n(H₀ true for all comparisons)', 
            fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(n_groups_list)
ax.set_ylim([0, 0.45])
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nConclusion: Without correction, the false positive rate grows dramatically.")
print("With just 3 groups (3 pairwise tests), the error rate is ~14% instead of 5%.")

## 4.4 One-Way ANOVA

**ANOVA (ANalysis Of VAriance)** is an omnibus test for comparing the means of three or more groups. It partitions the total variance into:

- **Between-group variance**: variation of group means around the grand mean
- **Within-group variance**: variation within each group

### Sums of squares and F-statistic

Let $\bar{X}$ be the grand mean (average across all observations) and $\bar{X}_j$ be the mean of group $j$. With $n_j$ observations in group $j$ and $k$ groups:

$$SS_{\text{between}} = \sum_{j=1}^{k} n_j (\bar{X}_j - \bar{X})^2$$

$$SS_{\text{within}} = \sum_{j=1}^{k} \sum_{i=1}^{n_j} (X_{ij} - \bar{X}_j)^2$$

$$SS_{\text{total}} = SS_{\text{between}} + SS_{\text{within}}$$

Mean squares are sums of squares divided by degrees of freedom:

$$MS_{\text{between}} = \frac{SS_{\text{between}}}{k - 1}, \quad MS_{\text{within}} = \frac{SS_{\text{within}}}{n - k}$$

The F-statistic is:

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}$$

Under $H_0$ (all group means equal), $F \sim F_{k-1, n-k}$.

In [ ]:
# One-way ANOVA: Income across three infrastructure types
# Interporto, Airport, Control regions

np.random.seed(42)

n_per_group = 15

# Generate data for three groups
interporto = np.random.normal(21, 3, n_per_group)  # Mean £21k
airport = np.random.normal(23, 3, n_per_group)    # Mean £23k
control = np.random.normal(29, 3, n_per_group)    # Mean £29k (highest)

all_data = np.concatenate([interporto, airport, control])
group_labels = ['Interporto']*n_per_group + ['Airport']*n_per_group + ['Control']*n_per_group

n_total = len(all_data)
k = 3  # number of groups
grand_mean = all_data.mean()

print("Descriptive Statistics:")
print(f"  Interporto: mean = £{interporto.mean():.2f}k, SD = £{interporto.std(ddof=1):.2f}k")
print(f"  Airport:    mean = £{airport.mean():.2f}k, SD = £{airport.std(ddof=1):.2f}k")
print(f"  Control:    mean = £{control.mean():.2f}k, SD = £{control.std(ddof=1):.2f}k")
print(f"  Grand mean: £{grand_mean:.2f}k\n")

# Manual ANOVA calculation
print("Manual ANOVA Calculation:")
print("="*60)

# SS_between
ss_between = (n_per_group * (interporto.mean() - grand_mean)**2 +
              n_per_group * (airport.mean() - grand_mean)**2 +
              n_per_group * (control.mean() - grand_mean)**2)
print(f"\nSS_between = {n_per_group} × ({interporto.mean():.4f} - {grand_mean:.4f})² + ...")
print(f"           = {ss_between:.4f}")

# SS_within
ss_within = (np.sum((interporto - interporto.mean())**2) +
             np.sum((airport - airport.mean())**2) +
             np.sum((control - control.mean())**2))
print(f"\nSS_within  = Σ(X_ij - X_j̄)² for all groups")
print(f"           = {ss_within:.4f}")

# SS_total
ss_total = np.sum((all_data - grand_mean)**2)
print(f"\nSS_total   = Σ(X_ij - X̄)²")
print(f"           = {ss_total:.4f}")
print(f"\nCheck: SS_between + SS_within = {ss_between + ss_within:.4f} ≈ {ss_total:.4f} ✓")

# Mean squares and F-statistic
df_between = k - 1
df_within = n_total - k
df_total = n_total - 1

ms_between = ss_between / df_between
ms_within = ss_within / df_within
f_stat = ms_between / ms_within

print(f"\nDegrees of freedom:")
print(f"  df_between = {k} - 1 = {df_between}")
print(f"  df_within  = {n_total} - {k} = {df_within}")
print(f"  df_total   = {n_total} - 1 = {df_total}")

print(f"\nMean Squares:")
print(f"  MS_between = {ss_between:.4f} / {df_between} = {ms_between:.4f}")
print(f"  MS_within  = {ss_within:.4f} / {df_within} = {ms_within:.4f}")

print(f"\nF-statistic:")
print(f"  F = {ms_between:.4f} / {ms_within:.4f} = {f_stat:.4f}")

# p-value from F-distribution
p_value = 1 - stats.f.cdf(f_stat, df_between, df_within)
print(f"  p-value = P(F_{df_between},{df_within} > {f_stat:.4f}) = {p_value:.6f}")
print(f"\n  Conclusion: {'Reject H₀' if p_value < 0.05 else 'Fail to reject H₀'} at 5% level")

print("\n" + "="*60)
print("Verification using scipy.stats.f_oneway:")
f_scipy, p_scipy = stats.f_oneway(interporto, airport, control)
print(f"  F = {f_scipy:.4f}")
print(f"  p-value = {p_scipy:.6f} ✓")

In [ ]:
# Visualisation: Groups with F-distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Box plots of the three groups
data_anova = pd.DataFrame({
    'Income (£k)': all_data,
    'Group': group_labels
})

sns.boxplot(data=data_anova, x='Group', y='Income (£k)', ax=axes[0], palette='Set2')
axes[0].set_title('Income by Infrastructure Type', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Income (£ thousands)')
axes[0].axhline(y=grand_mean, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Grand Mean')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].legend()

# Right panel: F-distribution with observed test statistic
x_f = np.linspace(0, 8, 500)
y_f = stats.f.pdf(x_f, df_between, df_within)

axes[1].plot(x_f, y_f, 'b-', linewidth=2, label=f'F({df_between},{df_within})')
axes[1].axvline(x=f_stat, color='red', linestyle='--', linewidth=2, label=f'Observed F = {f_stat:.4f}')

# Shade rejection region
critical_f = stats.f.ppf(0.95, df_between, df_within)
x_reject = x_f[x_f >= critical_f]
y_reject = stats.f.pdf(x_reject, df_between, df_within)
axes[1].fill_between(x_reject, y_reject, alpha=0.3, color='red', label='Rejection region (α=0.05)')

axes[1].set_xlabel('F-statistic')
axes[1].set_ylabel('Probability Density')
axes[1].set_title(f'F-Distribution: H₀ True\n(p-value = {p_value:.4f})', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4.5 Post-Hoc Tests

Once we reject the null hypothesis in ANOVA (concluding that not all means are equal), we need to determine **which pairs of groups differ**. This requires multiple comparisons adjustments to control the family-wise error rate.

### Bonferroni Correction

The simplest approach: divide the significance level by the number of comparisons. For $m = 3$ pairwise tests:
$$\alpha_{\text{adjusted}} = \frac{0.05}{3} \approx 0.017$$

Conservative but easy to understand.

### Tukey HSD (Honestly Significant Difference)

Based on the studentised range distribution. Generally more powerful than Bonferroni, especially with many groups. Designed specifically for pairwise comparisons in a balanced ANOVA.

The critical value is:
$$\text{HSD} = q_{\alpha}(k, df_{\text{within}}) \times SE$$

where $q$ is the critical value from the studentised range distribution and $SE = \sqrt{MS_{\text{within}} / n}$.

In [ ]:
# Post-hoc tests: Bonferroni and Tukey HSD

print("POST-HOC ANALYSIS")
print("="*70)

# Prepare data for statsmodels
df_test = pd.DataFrame({
    'income': all_data,
    'group': group_labels
})

# Bonferroni method
print("\n1. BONFERRONI CORRECTION")
print("-"*70)
n_comparisons = 3  # Interporto vs Airport, Interporto vs Control, Airport vs Control
alpha_bonf = 0.05 / n_comparisons
print(f"Number of pairwise comparisons: {n_comparisons}")
print(f"Adjusted significance level: {alpha_bonf:.4f}\n")

# Perform t-tests with Bonferroni correction
pairs = [('Interporto', 'Airport'),
         ('Interporto', 'Control'),
         ('Airport', 'Control')]

print("Pairwise t-tests (Bonferroni-corrected):")
print(f"{'Comparison':<30} {'t-stat':>10} {'p-value':>12} {'Significant':>12}")
print("-"*70)

for pair in pairs:
    group1 = df_test[df_test['group'] == pair[0]]['income'].values
    group2 = df_test[df_test['group'] == pair[1]]['income'].values
    t, p = stats.ttest_ind(group1, group2, equal_var=False)
    sig = "Yes" if p < alpha_bonf else "No"
    print(f"{pair[0]:12} vs {pair[1]:12} {t:10.4f} {p:12.6f} {sig:>12}")

# Tukey HSD
print("\n" + "="*70)
print("\n2. TUKEY HSD TEST")
print("-"*70)

tukey_result = pairwise_tukeyhsd(endog=df_test['income'], 
                                  groups=df_test['group'],
                                  alpha=0.05)

print(tukey_result)

print("\nInterpretation:")
print("  reject=True means the pairwise difference is significant at 5% level")
print("  meandiff: difference in group means")
print("  lower/upper: 95% confidence interval for the difference")

In [ ]:
# Visualise post-hoc results
fig, ax = plt.subplots(figsize=(10, 6))

# Extract Tukey results
tukey_df = pd.DataFrame(data=tukey_result.summary().data[1:], 
                        columns=tukey_result.summary().data[0])
tukey_df['meandiff'] = tukey_df['meandiff'].astype(float)
tukey_df['lower'] = tukey_df['lower'].astype(float)
tukey_df['upper'] = tukey_df['upper'].astype(float)

# Plot confidence intervals
for idx, row in tukey_df.iterrows():
    comparison = f"{row['group1']} vs {row['group2']}"
    ci_lower = row['lower']
    ci_upper = row['upper']
    mean_diff = row['meandiff']
    significant = row['reject']
    
    color = 'red' if significant else 'blue'
    ax.plot([ci_lower, ci_upper], [idx, idx], color=color, linewidth=2)
    ax.scatter([mean_diff], [idx], color=color, s=100, zorder=3, marker='o')
    ax.text(ci_upper + 1, idx, f"{'Sig.' if significant else 'NS'}", 
           va='center', fontsize=10, fontweight='bold')

ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_yticks(range(len(tukey_df)))
ax.set_yticklabels([f"{row['group1']} vs {row['group2']}" for _, row in tukey_df.iterrows()])
ax.set_xlabel('Difference in Mean Income (£ thousands)', fontsize=11, fontweight='bold')
ax.set_title('Tukey HSD Post-Hoc Test: 95% Confidence Intervals\n(Red = significant, Blue = not significant)', 
            fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 4.6 Assumptions & Non-Parametric Alternatives

Parametric tests (t-test, ANOVA) assume:
1. **Normality**: observations within each group are normally distributed
2. **Homogeneity of variances**: all groups have equal population variances ($\sigma_1^2 = \sigma_2^2 = \cdots$)
3. **Independence**: observations are independent

### Robustness

- **t-test and ANOVA are robust** to moderate departures from normality due to the Central Limit Theorem, especially with $n \geq 30$ per group
- Small departures from equal variances don't matter much; Welch's t-test (and Welch's ANOVA) don't assume equal variances
- Severe non-normality or extreme outliers → use **non-parametric alternatives**

### Non-Parametric Tests

**Mann-Whitney U test** (alternative to independent t-test):
- Tests whether two distributions are stochastically ordered
- Works on ranks, not raw data
- No normality assumption

**Kruskal-Wallis test** (alternative to one-way ANOVA):
- Extension of Mann-Whitney to $k \geq 2$ groups
- Tests whether the distributions of groups differ
- No normality assumption

In [ ]:
# Assumption checking: Levene's test and Shapiro-Wilk test

print("ASSUMPTION CHECKING")
print("="*70)

# 1. Levene's test for homogeneity of variances
print("\n1. LEVENE'S TEST FOR HOMOGENEITY OF VARIANCES")
print("-"*70)
levene_stat, levene_p = stats.levene(interporto, airport, control)
print(f"Levene's test statistic: {levene_stat:.4f}")
print(f"p-value: {levene_p:.6f}")
print(f"Conclusion: {'Reject H₀ (variances differ)' if levene_p < 0.05 else 'Fail to reject H₀ (variances equal)'}")

# 2. Shapiro-Wilk test for normality within each group
print("\n2. SHAPIRO-WILK TEST FOR NORMALITY (within each group)")
print("-"*70)

groups_dict = {'Interporto': interporto, 'Airport': airport, 'Control': control}
for group_name, group_data in groups_dict.items():
    sw_stat, sw_p = stats.shapiro(group_data)
    print(f"{group_name:12} W = {sw_stat:.4f}, p = {sw_p:.6f} ", end="")
    print(f"{'→ Not normal' if sw_p < 0.05 else '→ Normal'}")

print("\nNote: p > 0.05 suggests data are consistent with normality.")
print("      With moderate sample sizes, t-test/ANOVA are robust even if violated.")

In [ ]:
# Non-parametric alternatives on skewed data
print("\nNON-PARAMETRIC ALTERNATIVES")
print("="*70)

# Generate more skewed (lognormal) data to motivate non-parametric tests
np.random.seed(42)
n_skew = 20

# Lognormal distributions (right-skewed)
interporto_skew = np.random.lognormal(2.8, 0.4, n_skew)  # Skewed income data
airport_skew = np.random.lognormal(3.0, 0.4, n_skew)
control_skew = np.random.lognormal(3.3, 0.4, n_skew)

all_skew = np.concatenate([interporto_skew, airport_skew, control_skew])

print("\nUsing SKEWED (lognormal) income data:")
print(f"  Interporto: median = £{np.median(interporto_skew):.2f}, skewness = {stats.skew(interporto_skew):.3f}")
print(f"  Airport:    median = £{np.median(airport_skew):.2f}, skewness = {stats.skew(airport_skew):.3f}")
print(f"  Control:    median = £{np.median(control_skew):.2f}, skewness = {stats.skew(control_skew):.3f}")

# Parametric ANOVA (for comparison)
print("\n1. PARAMETRIC: One-Way ANOVA")
print("-"*70)
f_param, p_param = stats.f_oneway(interporto_skew, airport_skew, control_skew)
print(f"F-statistic: {f_param:.4f}")
print(f"p-value: {p_param:.6f}")
print(f"Conclusion: {'Reject H₀' if p_param < 0.05 else 'Fail to reject H₀'} at 5% level")

# Kruskal-Wallis test (non-parametric alternative)
print("\n2. NON-PARAMETRIC: Kruskal-Wallis Test")
print("-"*70)
h_stat, p_kw = stats.kruskal(interporto_skew, airport_skew, control_skew)
print(f"H-statistic: {h_stat:.4f}")
print(f"p-value: {p_kw:.6f}")
print(f"Conclusion: {'Reject H₀' if p_kw < 0.05 else 'Fail to reject H₀'} at 5% level")

print("\nNote: Both tests lead to the same conclusion in this example.")
print("      Kruskal-Wallis does not assume normality, only that distributions differ.")

# Mann-Whitney U test for pairwise comparisons (non-parametric)
print("\n3. PAIRWISE NON-PARAMETRIC: Mann-Whitney U Test")
print("-"*70)
print(f"{'Comparison':<30} {'U-stat':>10} {'p-value':>12}")
print("-"*70)

alpha_bonf_np = 0.05 / 3
for pair in pairs:
    if pair[0] == 'Interporto':
        g1 = interporto_skew
    elif pair[0] == 'Airport':
        g1 = airport_skew
    else:
        g1 = control_skew
    
    if pair[1] == 'Interporto':
        g2 = interporto_skew
    elif pair[1] == 'Airport':
        g2 = airport_skew
    else:
        g2 = control_skew
    
    u_stat, p_mw = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    print(f"{pair[0]:12} vs {pair[1]:12} {u_stat:10.1f} {p_mw:12.6f}")

In [ ]:
# Visualise normal vs skewed data
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Top row: Normal data (original)
groups_normal = [interporto, airport, control]
labels_normal = ['Interporto', 'Airport', 'Control']

for idx, (data, label) in enumerate(zip(groups_normal, labels_normal)):
    ax = axes[0, idx]
    ax.hist(data, bins=8, edgecolor='black', alpha=0.7, color='C0')
    ax.set_title(f'{label}\n(Normal)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency')
    ax.set_xlabel('Income (£ thousands)')
    ax.grid(True, alpha=0.3, axis='y')

# Bottom row: Skewed data (lognormal)
groups_skew = [interporto_skew, airport_skew, control_skew]

for idx, (data, label) in enumerate(zip(groups_skew, labels_normal)):
    ax = axes[1, idx]
    ax.hist(data, bins=8, edgecolor='black', alpha=0.7, color='C2')
    ax.set_title(f'{label}\n(Skewed)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency')
    ax.set_xlabel('Income (£ thousands)')
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Normal vs Skewed Distributions: When to Use Non-Parametric Tests', 
            fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("When to use non-parametric tests:")
print("  - Severe non-normality (especially outliers or heavy skewness)")
print("  - Small sample sizes (n < 30 per group) with clear non-normality")
print("  - When you have ranked or ordinal data")
print("  - If assumption violations appear egregious")

## Exercises

### Exercise 1: Two-Sample t-Test
Two regional economies' average annual household income:
- Region A: n=25, mean=£48.2k, SD=£9.5k
- Region B: n=22, mean=£51.7k, SD=£10.3k

Test at the 5% level whether the regions have significantly different mean incomes. Calculate the t-statistic, p-value, and 95% confidence interval for the difference.

### Exercise 2: Paired t-Test
Property assessments in 16 municipalities before and after a transport policy change. You are given before and after values (in £100k):
```
Before: [2.1, 2.3, 2.0, 1.9, 2.2, 1.8, 2.4, 2.0,
         2.1, 2.3, 1.9, 2.5, 2.2, 2.0, 2.1, 1.8]
After:  [2.0, 2.2, 2.1, 1.9, 2.3, 1.7, 2.5, 2.1,
         2.0, 2.4, 2.0, 2.6, 2.3, 2.1, 2.2, 1.9]
```

Perform a paired t-test. Interpret the result: did the policy have a significant effect?

### Exercise 3: ANOVA with Post-Hoc Analysis
Manufacturing firms in four size categories have average annual revenues (£ millions):
- Small (n=12): mean = £2.3, SD = £0.5
- Medium (n=15): mean = £8.5, SD = £1.2
- Large (n=10): mean = £25.8, SD = £3.1
- Multinational (n=8): mean = £145.2, SD = £15.0

Perform a one-way ANOVA. If significant, apply Tukey HSD to identify which pairs differ. What does this suggest about firm size and revenue?

### Exercise 4: Parametric vs Non-Parametric
You have income data from three regions with heavy outliers (e.g., billionaires). The Shapiro-Wilk test rejects normality (p < 0.01) in all three groups.

Discuss: Why might the parametric t-test or ANOVA still be reasonable? When should you switch to Mann-Whitney or Kruskal-Wallis? What practical considerations matter?

In [ ]:
# EXERCISE STARTER CODE
# Uncomment and complete each exercise

# ============================================================================
# EXERCISE 1: Two-Sample t-Test
# ============================================================================

# Summary statistics given (no raw data):
# Region A: n=25, mean=48.2, sd=9.5
# Region B: n=22, mean=51.7, sd=10.3

# TODO: Calculate the t-statistic and p-value manually
# Hint: SE_diff = sqrt(s_A^2/n_A + s_B^2/n_B)

# ============================================================================
# EXERCISE 2: Paired t-Test
# ============================================================================

before_ex = np.array([2.1, 2.3, 2.0, 1.9, 2.2, 1.8, 2.4, 2.0,
                      2.1, 2.3, 1.9, 2.5, 2.2, 2.0, 2.1, 1.8])
after_ex = np.array([2.0, 2.2, 2.1, 1.9, 2.3, 1.7, 2.5, 2.1,
                     2.0, 2.4, 2.0, 2.6, 2.3, 2.1, 2.2, 1.9])

# TODO: Perform paired t-test and interpret
# Hint: Calculate differences, then use scipy.stats.ttest_rel()

# ============================================================================
# EXERCISE 3: ANOVA with Post-Hoc
# ============================================================================

# Given summary statistics (no raw data):
# Small:        n=12, mean=2.3, sd=0.5
# Medium:       n=15, mean=8.5, sd=1.2
# Large:        n=10, mean=25.8, sd=3.1
# Multinational:n=8, mean=145.2, sd=15.0

# TODO: Generate synthetic data with these properties and run ANOVA + Tukey
# Hint: Use np.random.normal() with the given means and SDs

# ============================================================================
# EXERCISE 4: Discussion
# ============================================================================

# TODO: Write a brief discussion (as text) about when to use parametric vs
# non-parametric tests. Consider sample size, severity of non-normality,
# and practical robustness.

print("\n" + "="*70)
print("Exercise starter code above. Complete each exercise, test your code,")
print("and interpret the results in the context of regional economics.")
print("="*70)